# TCGA-ACC: extracción de datos de RNA-Seq y selección de 10 genes por varianza

Taller de Oncología Traslacional (TCGA) | Healtech / Analítica de Datos en Salud | Universidad Externado de Colombia

Docente: Eliana González Artunduaga

Integrantes: David Lobo, David Quijano, David Duarte

In [ ]:
import os, re, math, json, time, requests
import numpy as np, pandas as pd

API = "https://api.gdc.cancer.gov"
PROYECTO = "TCGA-ACC"
DATA_DIR = "gdc_acc_star"
os.makedirs(DATA_DIR, exist_ok=True)

## 1. Por qué se eligió el carcinoma adrenocortical (TCGA-ACC)

El carcinoma adrenocortical (ACC) es un tumor maligno que se origina en la corteza de la glándula suprarrenal, es decir, en el tejido que produce las hormonas esteroideas (cortisol, aldosterona y andrógenos). Se trata de una neoplasia poco frecuente: en un artículo publicado en Endocrine se estima una incidencia de alrededor de un caso por millón de personas al año en adultos, y en el registro de cáncer de los Países Bajos (359 pacientes entre 1993 y 2010) la tasa estandarizada por edad pasó de 1.3 a 1.0 por millón de personas-año. Por esto, el ACC no tiene el peso epidemiológico de un cáncer como el de mama o el de pulmón, pero sí tiene una relevancia clínica alta.

Esta relevancia viene del pronóstico. En el mismo registro neerlandés la supervivencia mediana fue de 159 meses en los estadios I y II, de 26 meses en el estadio III y de apenas 5 meses en el estadio IV, y los autores no encontraron una mejora de la supervivencia durante el periodo estudiado a pesar de que aumentó el número de cirugías. A esto se suma que una parte importante de los tumores es funcionante: en el artículo de Endocrine el 63% de los pacientes tenía un exceso hormonal verificado bioquímicamente, siendo el hipercortisolismo aislado el más frecuente (41%). Es decir, el tumor no solo crece, sino que además altera el equilibrio hormonal del paciente.

Con respecto a la biología, el proyecto TCGA caracterizó 91 tumores primarios de ACC (Zheng et al., Cancer Cell, 2016). Entre los genes alterados con más frecuencia están TP53 (21% de los casos, ya sea por mutación, alteración del número de copias o metilación), ZNRF3 (19%), CTNNB1 (16%) y CDKN2A (15%). Ese estudio también agrupó los tumores en tres grupos moleculares (CoC I, II y III) cuya tasa de progresión de la enfermedad fue muy distinta (7%, 56% y 96%), y definió un puntaje de diferenciación adrenocortical construido con 25 genes de la corteza (entre ellos, enzimas esteroidogénicas), el cual fue mayor en los tumores funcionantes. Estos hallazgos indican que el ACC es un cáncer muy heterogéneo, y esta heterogeneidad es la que se espera ver reflejada en los genes con mayor varianza.

Por último, hay una razón práctica para la elección. TCGA-ACC es una de las cohortes más pequeñas del programa (92 casos en el GDC, 79 con RNA-Seq), lo que hace que la descarga y el manejo de la matriz sean livianos, y que al mismo tiempo el problema de alta dimensionalidad del taller sea muy marcado (cerca de 60.000 genes para 79 pacientes). La limitación de este argumento es que las cifras de incidencia y supervivencia citadas vienen de registros de otros países, y no se revisaron cifras para Colombia, por lo que no se pueden extrapolar directamente.

## 2. Metadata del proyecto en el GDC

La ficha del proyecto se consulta directamente al endpoint /projects de la API, pidiendo el resumen de las categorías de datos y de las estrategias experimentales. Con esto se obtiene la sigla del proyecto, el órgano de origen y el número de casos, que son los datos de identificación que pide el taller.

In [ ]:
r = requests.get(f"{API}/projects/{PROYECTO}",
                 params={"expand": "summary,summary.data_categories,summary.experimental_strategies,program"}).json()["data"]
print("Proyecto:", r["project_id"], "|", r["name"], "| primary_site:", r.get("primary_site"))
print("case_count:", r["summary"]["case_count"], "| file_count:", r["summary"]["file_count"])
print("Categorías de datos:", {d["data_category"]: d["case_count"] for d in r["summary"].get("data_categories", [])})
print("Estrategias:", {d["experimental_strategy"]: d["case_count"] for d in r["summary"].get("experimental_strategies", [])})

Proyecto: TCGA-ACC | Adrenocortical Carcinoma | primary_site: ['Adrenal gland']
case_count: 92 | file_count: 5792
Categorías de datos: {'Simple Nucleotide Variation': 92, 'Sequencing Reads': 92, 'Biospecimen': 92, 'Clinical': 92, 'Copy Number Variation': 92, 'Transcriptome Profiling': 80, 'DNA Methylation': 80, 'Proteome Profiling': 46, 'Somatic Structural Variation': 74, 'Structural Variation': 84}
Estrategias: {'RNA-Seq': 79, 'miRNA-Seq': 80, 'WXS': 92, 'WGS': 74, 'ATAC-Seq': 9, 'Genotyping Array': 92, 'Methylation Array': 80, 'Reverse Phase Protein Array': 46, 'Diagnostic Slide': 56, 'Tissue Slide': 92}


La cohorte es TCGA-ACC (Adrenocortical Carcinoma), con la glándula suprarrenal como sitio primario y 92 casos en total. Estos 92 casos tienen datos clínicos, de exoma (WXS) y de variación en el número de copias, pero no todos tienen transcriptoma: la categoría Transcriptome Profiling cubre 80 casos y dentro de esta la estrategia RNA-Seq cubre 79 (la diferencia de un caso se explica porque la categoría también incluye miRNA-Seq, con 80 casos). Por lo tanto, el tamaño muestral de este trabajo va a ser N = 79 pacientes y no 92, ya que la expresión génica solo existe para esos 79.

## 3. Por qué Python y Google Colab

El entorno elegido es Python dentro de Google Colab. El script de referencia del taller está en R y usa TCGAbiolinks, un paquete que envuelve la API del GDC y entrega el resultado como un objeto SummarizedExperiment. Este enfoque es más corto, sin embargo oculta lo que pasa en cada paso (qué filtros se mandan, qué archivos llegan, cómo se arma la matriz), y para un taller en donde se pide explicar cómo el código interactúa con la API resultaba más conveniente hacer las peticiones de forma explícita con la librería requests y construir la matriz con pandas.

Con respecto a la memoria, la matriz completa tiene 60.660 genes por 79 pacientes (enteros de 64 bits) y ocupa cerca de 38 MB, por lo que cabe sin problema en una sesión estándar de Colab. El cuello de botella no es la memoria sino la descarga: son 79 archivos independientes, y por esto la función de descarga guarda cada archivo en disco y salta los que ya existen, de modo que si la sesión se interrumpe no se empieza de cero. Con respecto a la reproducibilidad, el notebook se ejecuta de arriba hacia abajo sin pasos manuales, todas las decisiones (umbrales, exclusiones) quedan escritas como código y, por ser Python, la matriz queda lista para pasar a scikit-learn si más adelante se quiere modelar.

Cabe aclarar que R no es una mala opción para este problema, y que paquetes como DESeq2 o edgeR están pensados para R. Aquí no se necesitaron porque solo se calculan CPM y varianzas, no se hace expresión diferencial. El problema con Colab es que el disco de la sesión es temporal: si no se monta Google Drive en DATA_DIR, los archivos descargados se pierden al cerrar la sesión, y como las versiones de las librerías no quedan fijadas podrían traer diferencias menores si el notebook se corre meses después.

## 4. Consulta de los archivos a la API del GDC

La consulta se hace con un POST al endpoint /files, mandando un filtro con cinco condiciones: el proyecto (TCGA-ACC), la estrategia experimental (RNA-Seq), el tipo de dato (Gene Expression Quantification), el flujo de trabajo (STAR - Counts) y el acceso abierto. De cada archivo se piden el identificador, el nombre y los datos del paciente y de la muestra, ya que después estos datos permiten nombrar las columnas de la matriz con el código del paciente y verificar que no haya pacientes repetidos.

In [ ]:
def filtro(campo, valor):
    return {"op": "in", "content": {"field": campo, "value": [valor]}}

body = {
    "filters": {"op": "and", "content": [
        filtro("cases.project.project_id", PROYECTO),
        filtro("experimental_strategy", "RNA-Seq"),
        filtro("data_type", "Gene Expression Quantification"),
        filtro("analysis.workflow_type", "STAR - Counts"),
        filtro("access", "open"),
    ]},
    "fields": "file_id,file_name,file_size,cases.case_id,cases.submitter_id,cases.samples.sample_type,cases.samples.submitter_id,analysis.workflow_type",
    "size": 500, "format": "JSON",
}
hits = requests.post(f"{API}/files", json=body).json()["data"]["hits"]
meta = pd.DataFrame([{
    "file_id": h["file_id"], "file_name": h["file_name"],
    "case_id": h["cases"][0]["case_id"], "paciente": h["cases"][0]["submitter_id"],
    "muestra": h["cases"][0]["samples"][0]["submitter_id"],
    "tipo_muestra": h["cases"][0]["samples"][0]["sample_type"],
    "n_muestras_en_archivo": len(h["cases"][0]["samples"]),
} for h in hits])

print("Archivos:", len(meta), "| pacientes únicos:", meta["paciente"].nunique())
print(meta["tipo_muestra"].value_counts().to_string())
dup = meta[meta.duplicated("paciente", keep=False)]
print("Pacientes con más de un archivo:", dup["paciente"].nunique())
if len(dup): display(dup.sort_values("paciente"))

Archivos: 79 | pacientes únicos: 79
tipo_muestra
Primary Tumor    79
Pacientes con más de un archivo: 0


Se obtuvieron 79 archivos para 79 pacientes distintos, todos de tumor primario, por lo que no hay pacientes con más de un archivo y no se debe de elegir entre aliquots. Cabe aclarar que esto también implica que la cohorte no tiene tejido normal de referencia, algo que más adelante limita la interpretación de los genes seleccionados.

## 5. Descarga y ensamblaje de la matriz

La descarga se hace archivo por archivo con el endpoint /data/{file_id}. La función escribe primero en un archivo temporal (.part) y solo lo renombra cuando la descarga termina completa, y si hay un error de red reintenta hasta seis veces con una espera cada vez mayor. Esto evita que un archivo cortado quede guardado como si estuviera bien.

In [ ]:
def descargar(fid, intentos=6):
    ruta = os.path.join(DATA_DIR, f"{fid}.tsv")
    if os.path.exists(ruta) and os.path.getsize(ruta) > 0:
        return ruta
    ultimo = None
    for i in range(intentos):
        try:
            with requests.get(f"{API}/data/{fid}", stream=True, timeout=(15, 300)) as resp:
                resp.raise_for_status()
                with open(ruta + ".part", "wb") as f:
                    for trozo in resp.iter_content(1 << 16):
                        f.write(trozo)
            os.replace(ruta + ".part", ruta)
            return ruta
        except requests.exceptions.RequestException as e:
            ultimo = e
            time.sleep(3 * (i + 1))
    raise RuntimeError(f"No se pudo descargar {fid}: {ultimo}")

for i, fid in enumerate(meta["file_id"], 1):
    descargar(fid)
    if i % 10 == 0: print(f"{i}/{len(meta)}")

10/79
20/79
30/79
40/79
50/79
60/79
70/79


Cada archivo STAR - Counts es una tabla con una fila por gen (identificador Ensembl, nombre y tipo de gen) y varias columnas de cuantificación. Se usa la columna unstranded, que son los conteos crudos de lecturas por gen, y se descartan las filas resumen de STAR (las que empiezan por N_, como las lecturas sin mapear) ya que no son genes. Los 79 vectores se unen en un DataFrame con los genes en las filas y los pacientes en las columnas.

In [ ]:
# Si un paciente tiene varios archivos, se conserva el primero (la tabla de la sección 4 muestra que aquí no ocurre)
meta_u = meta.drop_duplicates("paciente").reset_index(drop=True)

series, anot = {}, None
for _, fila in meta_u.iterrows():
    d = pd.read_csv(os.path.join(DATA_DIR, f"{fila.file_id}.tsv"), sep="\t", comment="#")
    d = d[~d["gene_id"].str.startswith("N_")]            # filas resumen de STAR
    if anot is None:
        anot = d[["gene_id", "gene_name", "gene_type"]].set_index("gene_id")
    series[fila.paciente] = d.set_index("gene_id")["unstranded"]
conteos = pd.DataFrame(series)
print("Matriz:", conteos.shape, "| dtype:", conteos.values.dtype,
      "| tamaño ≈", round(conteos.values.nbytes/1e6, 1), "MB")
print("P/N =", round(conteos.shape[0] / conteos.shape[1], 1))

Matriz: (60660, 79) | dtype: int64 | tamaño ≈ 38.3 MB
P/N = 767.8


La matriz cruda tiene 60.660 genes y 79 pacientes, lo que da una relación P/N de 767.8, es decir, hay casi 768 variables por cada paciente. En términos simples, es como tener un sistema con muchas más incógnitas que ecuaciones: con tantas variables un modelo puede encontrar patrones que separen a los pacientes por pura casualidad, haciendo que se sobreajuste (overfitting) y no generalice. Por esto el taller pide reducir las dimensiones antes de modelar, y lo que se hace en las siguientes secciones es justamente eso.

## 6. Filtro de genes poco expresados

El primer paso de la reducción es quitar los genes que casi no se expresan en esta cohorte. Un gen con conteos de 0 o 1 en casi todos los pacientes no aporta información sobre la diferencia entre tumores, y además tiene una varianza pequeña pero ruidosa. El criterio que se usó es conservar los genes con al menos 10 conteos en al menos el 20% de los pacientes (16 de 79). Este umbral es una decisión práctica, parecida a la idea de la función filterByExpr de edgeR (que también parte de un mínimo de conteos), pero no es el único posible: umbrales más estrictos dejarían menos genes y umbrales más flexibles dejarían más ruido.

En la misma celda se descarga la anotación GENCODE v36 (cuyos identificadores coinciden con los de la matriz) para saber en qué cromosoma está cada gen, lo que se necesita más adelante para tratar los genes de los cromosomas sexuales.

In [ ]:
N = conteos.shape[1]
min_pac = math.ceil(0.20 * N)
pasa = (conteos >= 10).sum(axis=1) >= min_pac
es_mt = anot.loc[conteos.index, "gene_name"].str.startswith("MT-")
print(f"Umbral: >=10 conteos en >= {min_pac} de {N} pacientes")
print("Genes que pasan el filtro:", int(pasa.sum()))
print("  de ellos MT-:", int((pasa & es_mt).sum()))
print("  protein_coding entre los que pasan:", int((anot.loc[conteos.index[pasa], "gene_type"] == "protein_coding").sum()))

# Genes de cromosomas sexuales: se descarga la anotación GENCODE v36 (~45 MB) para ubicar cada gen
USAR_SEXO = True
sexo = pd.Series(False, index=conteos.index)
if USAR_SEXO:
    url = "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_36/gencode.v36.annotation.gtf.gz"
    gtf = pd.read_csv(url, sep="\t", comment="#", header=None, usecols=[0, 2, 8], names=["chr", "feat", "attr"])
    gtf = gtf[gtf.feat == "gene"]
    gtf["gene_id"] = gtf.attr.str.extract(r'gene_id "([^"]+)"')
    ids_sexo = set(gtf.loc[gtf.chr.isin(["chrX", "chrY"]), "gene_id"])
    sexo = conteos.index.to_series().isin(ids_sexo)
    print("Genes X/Y en la matriz:", int(sexo.sum()), "| que pasan el filtro:", int((sexo & pasa).sum()))

Umbral: >=10 conteos en >= 16 de 79 pacientes
Genes que pasan el filtro: 21213
  de ellos MT-: 22
  protein_coding entre los que pasan: 15573
Genes X/Y en la matriz: 2990 | que pasan el filtro: 748


Pasan el filtro 21.213 genes de los 60.660 (alrededor del 35%), de los cuales 15.573 son codificantes de proteínas y 22 son mitocondriales (MT-). Con respecto a los cromosomas sexuales, hay 2.990 genes en X o Y en la matriz y 748 de ellos pasan el filtro. Estas cifras son el insumo para las exclusiones que se discuten en las secciones 7 y 8.

## 7. Varianza como criterio de selección

Para escoger los 10 genes se usa la varianza de cada gen entre los pacientes: un gen con varianza alta es un gen cuya expresión cambia mucho de un tumor a otro, por lo tanto diferencia a los pacientes y no se comporta igual en todos (como lo hacen los genes de mantenimiento celular). Se asume que estos genes son los más representativos de la heterogeneidad de la enfermedad, aunque como se aclara en la sección 12 esto no es lo mismo que ser los más asociados con el pronóstico.

La varianza no se calcula sobre los conteos crudos sino sobre log2(CPM + 1), en donde CPM son los conteos por millón de lecturas de cada paciente. Se divide por el tamaño de la librería porque cada paciente se secuenció con una profundidad distinta (sin esto, un paciente con más lecturas parecería tener todos sus genes más expresados), y se aplica el logaritmo porque en datos de conteos la varianza crece con el nivel de expresión, por lo que los genes muy abundantes dominarían cualquier ranking. El +1 evita el logaritmo de cero.

Para ver cuánto depende el resultado de las decisiones de preprocesamiento, la siguiente celda calcula el top 10 en cuatro escenarios: (A) filtro y sin genes MT- ni de cromosomas sexuales, (B) igual que A pero calculando el tamaño de la librería solo con los genes que quedan, (C) solo el filtro, sin excluir nada más y (D) filtro y sin MT-, conservando los genes sexuales.

In [ ]:
def log2cpm(m, lib=None):
    lib = m.sum(axis=0) if lib is None else lib
    return np.log2(m / lib * 1e6 + 1)

def top_var(m, k=10):
    return m.var(axis=1, ddof=1).sort_values(ascending=False).head(k)

def nombres(serie):
    return serie.rename(index=anot["gene_name"].to_dict())

lib_total = conteos.sum(axis=0)                       # tamaño de librería con TODOS los genes
excl = pasa & ~es_mt & ~sexo                          # genes que pasan el filtro, sin MT- y sin cromosomas sexuales

esc = {
    "A. Filtro, sin MT- ni cromosomas sexuales (lib = todos los genes)": log2cpm(conteos[excl], lib_total),
    "B. Igual que A, pero lib = solo los genes que quedan":                       log2cpm(conteos[excl]),
    "C. Solo filtro, sin excluir MT- ni cromosomas sexuales":                               log2cpm(conteos[pasa], lib_total),
    "D. Filtro y sin MT-, con genes sexuales":                          log2cpm(conteos[pasa & ~es_mt], lib_total),
}
resultados = {k: nombres(top_var(v)) for k, v in esc.items()}
for k, s in resultados.items():
    print("\n" + k); print(s.round(2).to_string())


A. Filtro, sin MT- ni cromosomas sexuales (lib = todos los genes)
gene_id
MEG3       15.16
CYP11B1    14.82
GSTA1      14.70
HSD3B2     12.52
KLK1       11.50
SLPI       11.33
CYP17A1    10.75
CCN3       10.34
NPTX1      10.23
DLK1       10.14

B. Igual que A, pero lib = solo los genes que quedan
gene_id
GSTA1      15.93
MEG3       15.39
CYP11B1    15.11
KLK1       12.91
SLPI       12.75
HSD3B2     12.48
CSDC2      11.40
BIRC7      10.90
RHCG       10.85
CALB1      10.73

C. Solo filtro, sin excluir MT- ni cromosomas sexuales
gene_id
MEG3       15.16
CYP11B1    14.82
GSTA1      14.70
HSD3B2     12.52
KLK1       11.50
SLPI       11.33
RPS4Y1     11.04
CYP17A1    10.75
FATE1      10.44
CCN3       10.34

D. Filtro y sin MT-, con genes sexuales
gene_id
MEG3       15.16
CYP11B1    14.82
GSTA1      14.70
HSD3B2     12.52
KLK1       11.50
SLPI       11.33
RPS4Y1     11.04
CYP17A1    10.75
FATE1      10.44
CCN3       10.34


Los mismos seis genes (MEG3, CYP11B1, GSTA1, HSD3B2, KLK1 y SLPI) ocupan los seis primeros lugares en los cuatro escenarios, aunque en el B cambia el orden, por lo que se pueden considerar resultados estables. Las diferencias aparecen del séptimo lugar en adelante. En los escenarios C y D entra RPS4Y1 (varianza 11.04), un gen del cromosoma Y, lo cual no es biología tumoral sino el sexo del paciente (es un gen alto en hombres y casi nulo en mujeres), y esto justifica excluir genes del Y. Por otra parte, en el escenario A, en donde se excluyen todos los genes de X y Y, desaparece FATE1 (que está en el cromosoma X) y entran DLK1 y NPTX1, por lo que excluir ambos cromosomas completos sacaría de la lista un gen con literatura directa en ACC solo por su ubicación. Por esto la exclusión se va a limitar al cromosoma Y.

En el escenario B la lista cambia más (entran CSDC2, BIRC7, RHCG y CALB1, y salen CYP17A1, CCN3, NPTX1 y DLK1). La explicación es que el tamaño de la librería debe reflejar la profundidad real de secuenciación de cada paciente, y no el tamaño de lo que quede después de filtrar: los genes mitocondriales ocupan una fracción enorme y muy variable de las lecturas (como se ve en la siguiente celda), así que quitarlos antes de normalizar cambia la escala de cada paciente de forma desigual. Por tanto, se usa el tamaño de librería con todos los genes.

Como comparación, la siguiente celda calcula la varianza sobre los conteos crudos. Esto sirve para justificar el uso de log2(CPM + 1): si no se transforma, la varianza de un gen depende de su nivel de expresión y no de qué tan diferente es entre pacientes.

In [ ]:
# Top de varianza con conteos crudos y magnitud de los conteos
crudo_todos = nombres(top_var(conteos, 10))
print("Varianza cruda, TODOS los genes:"); print(crudo_todos.map("{:.2e}".format).to_string())
print("\nConteo medio y rango de los 5 primeros:")
for g in crudo_todos.index[:5]:
    fila = conteos.loc[anot.index[anot.gene_name == g][0]]
    print(f"  {g}: media={fila.mean():,.0f}  min={fila.min():,.0f}  max={fila.max():,.0f}")
n_mt5 = sum(str(g).startswith("MT-") for g in crudo_todos.index[:5])
print(f"\nGenes MT- entre los 5 primeros: {n_mt5}")

# ¿Aparecen en el top 10 de log2CPM si no se excluyen antes?
top_c = list(resultados["C. Solo filtro, sin excluir MT- ni cromosomas sexuales"].index)
print("MT- en el top 10 de log2CPM sin excluirlos:", [g for g in top_c if str(g).startswith("MT-")] or "ninguno")

Varianza cruda, TODOS los genes:
gene_id
MT-RNR2    2.12e+12
MT-CO1     1.45e+12
MT-ND4     8.09e+11
MT-CO3     4.50e+11
MT-CO2     2.65e+11
DLK1       2.05e+11
MT-CYB     1.50e+11
MT-ND1     1.05e+11
MT-ND2     9.80e+10
MT-RNR1    6.97e+10

Conteo medio y rango de los 5 primeros:
  MT-RNR2: media=1,754,420  min=145,152  max=6,900,938
  MT-CO1: media=1,631,225  min=79,776  max=5,816,372
  MT-ND4: media=1,257,518  min=125,862  max=5,373,940
  MT-CO3: media=957,568  min=141,746  max=3,451,227
  MT-CO2: media=629,452  min=72,743  max=2,625,877

Genes MT- entre los 5 primeros: 5
MT- en el top 10 de log2CPM sin excluirlos: ninguno


Con los conteos crudos, los cinco primeros lugares son genes mitocondriales (MT-RNR2, MT-CO1, MT-ND4, MT-CO3 y MT-CO2), con conteos medios por paciente de entre 629.452 y 1.754.420 y un rango muy amplio (MT-RNR2 va de 145.152 a 6.900.938). Estos valores dicen más de cuánto RNA mitocondrial tenía cada muestra (lo que puede depender de factores técnicos y del tipo de célula) que de la biología que interesa. El primer gen no mitocondrial en este ranking es DLK1, en sexto lugar. Con log2CPM, en cambio, ningún gen MT- aparece en el top 10 aun sin excluirlos, por lo que la exclusión de estos genes no cambia la lista final y se mantiene solo como medida de precaución.

## 8. Genes del cromosoma Y

Como se vio en la sección anterior, RPS4Y1 entra al top 10 cuando se conservan los genes del cromosoma Y. Un gen del Y se expresa solamente en pacientes de sexo masculino, por lo tanto su varianza refleja el sexo del paciente y no la biología del tumor. Las siguientes celdas revisan cuántos genes del Y pasan el filtro y qué tipo de genes son, y cuántos genes quedarían según distintos criterios de exclusión.

In [ ]:
ids_y = set(gtf.loc[gtf.chr == "chrY", "gene_id"])
y = conteos.index.to_series().isin(ids_y)
print("Genes Y que pasan el filtro:", int((y & pasa).sum()))
print("Filtro sin MT ni Y:", int((pasa & ~es_mt & ~y).sum()))
print(nombres(top_var(log2cpm(conteos[pasa & ~es_mt & ~y], lib_total), 10)).round(2).to_string())

Genes Y que pasan el filtro: 19
Filtro sin MT ni Y: 21172
gene_id
MEG3       15.16
CYP11B1    14.82
GSTA1      14.70
HSD3B2     12.52
KLK1       11.50
SLPI       11.33
CYP17A1    10.75
FATE1      10.44
CCN3       10.34
NPTX1      10.23


In [ ]:
y_pasan = conteos.index[y & pasa]
d = anot.loc[y_pasan, ["gene_name", "gene_type"]].copy()
d["pacientes_con_>=10"] = (conteos.loc[y_pasan] >= 10).sum(axis=1)
print(d.sort_values("gene_type").to_string())

tipo = anot.loc[conteos.index, "gene_type"]
nom = anot.loc[conteos.index, "gene_name"]
variantes = {
    "Y protein_coding":  y & (tipo == "protein_coding"),
    "Y sin lncRNA":      y & (tipo != "lncRNA"),
    "Y + XIST":          y | (nom == "XIST"),
}
for k, m in variantes.items():
    print(k, "->", int((pasa & ~es_mt & ~m).sum()))

                     gene_name                           gene_type  pacientes_con_>=10
gene_id                                                                               
ENSG00000176728.10      TTTY14                              lncRNA                  17
ENSG00000260197.1   AC010889.1                              lncRNA                  18
ENSG00000278847.1   AC006157.1                              lncRNA                  16
ENSG00000215414.4      PSMA6P1                processed_pseudogene                  62
ENSG00000235001.3     EIF4A1P2                processed_pseudogene                  20
ENSG00000099725.14        PRKY                      protein_coding                  29
ENSG00000114374.13       USP9Y                      protein_coding                  31
ENSG00000129824.16      RPS4Y1                      protein_coding                  43
ENSG00000154620.6       TMSB4Y                      protein_coding                  16
ENSG00000165246.14      NLGN4Y             

Pasan el filtro 19 genes del cromosoma Y. Diez de ellos son codificantes (PRKY, USP9Y, RPS4Y1, TMSB4Y, NLGN4Y, UTY, EIF1AY, DDX3Y, ZFY y KDM5D) y se detectan con al menos 10 conteos en entre 16 y 43 pacientes (RPS4Y1 es el que más, con 43), un patrón que es compatible con que solo una parte de la cohorte sea de sexo masculino, aunque en este notebook no se cruzó con la variable clínica de sexo. Los otros nueve son lncRNA y pseudogenes, y entre estos hay dos que se detectan en la mayoría de la cohorte (PSMA6P1 en 62 pacientes y AC134878.2 en 75), lo que indica que no son específicos del sexo masculino (probablemente por secuencias duplicadas que también se alinean en otras regiones del genoma, algo que no se verificó aquí). Por esto no tiene mucho sentido excluir todos los genes del Y en bloque.

Con respecto a los criterios de exclusión, quitar solo los codificantes del Y deja 21.181 genes, quitar los que no son lncRNA deja 21.175, quitar los codificantes más XIST deja 21.171 y quitar todos los del Y (celda anterior) deja 21.172. Se adopta el primero (exclusión de los genes codificantes del Y). Cabe aclarar que es una decisión parcialmente arbitraria, y lo que se comprueba en la siguiente sección es que el top 10 no cambia si se excluyen todos los genes del Y, por lo que la decisión afecta el número de genes que quedan pero no la lista final.

## 9. Selección final de los 10 genes

Con las decisiones anteriores, el filtro final queda así: se conservan los genes con al menos 10 conteos en al menos 16 pacientes, se excluyen los genes mitocondriales (MT-) y se excluyen los genes codificantes del cromosoma Y. A los genes que quedan se les calcula log2(CPM + 1) con el tamaño de librería total de cada paciente, se ordenan por varianza y se toman los 10 primeros. Al final de la celda se comprueba que la lista no cambie si se excluyen todos los genes del Y.

In [ ]:
# Filtro final: expresión mínima, sin genes MT- y sin genes codificantes del cromosoma Y
y_codif = y & (tipo == "protein_coding")
final = pasa & ~es_mt & ~y_codif

print("Genes que se conservan:", int(final.sum()))
print("  excluidos por baja expresión:", int((~pasa).sum()))
print("  excluidos por ser MT-:", int((pasa & es_mt).sum()))
print("  excluidos por ser codificantes del Y:", int((pasa & ~es_mt & y_codif).sum()))

# log2(CPM+1) con el tamaño de librería calculado con todos los genes
expr = log2cpm(conteos[final], lib_total)
varianza = expr.var(axis=1, ddof=1).sort_values(ascending=False)
top10 = varianza.head(10).index

# Comprobación: ¿cambia la lista si se excluyen todos los genes del Y?
top10_todoY = top_var(log2cpm(conteos[pasa & ~es_mt & ~y], lib_total), 10).index
print("\nTop 10 (varianza en log2CPM):")
print(nombres(varianza.head(10)).round(2).to_string())
print("\n¿Misma lista si se excluyen todos los genes del Y?", set(top10) == set(top10_todoY))

Los 10 genes seleccionados son MEG3, CYP11B1, GSTA1, HSD3B2, KLK1, SLPI, CYP17A1, FATE1, CCN3 y NPTX1, con varianzas que van de 15.16 (MEG3) a 10.23 (NPTX1). Una varianza de 15 en escala log2 equivale a una desviación estándar de casi 4 unidades, es decir, que entre pacientes la expresión de estos genes cambia en promedio por un factor cercano a 15 (2 elevado a 3.9), y aun el último de la lista (10.23) cambia por un factor de alrededor de 9. Esto es mucha variación para un grupo de 79 tumores del mismo tipo.

Además, tres de los diez genes (CYP11B1, HSD3B2 y CYP17A1) son enzimas directas de la síntesis de esteroides, lo cual es coherente con lo que se esperaba de un tumor derivado de la corteza suprarrenal en el que unos pacientes conservan la función hormonal y otros no. La interpretación de cada gen se hace en la sección 11.

## 10. Matriz final

Se arma la matriz de 10 genes por 79 pacientes en dos versiones: una con log2(CPM + 1), que es la que se usaría para modelar, y otra con los conteos crudos de los mismos genes, por si se quiere hacer otro tipo de análisis. La celda imprime las dimensiones en cada etapa de la reducción (que es la evidencia que pide el taller), muestra la tabla de los 10 genes y guarda las dos matrices en CSV dentro de la carpeta resultados.

In [ ]:
mat_log = expr.loc[top10]          # 10 genes x 79 pacientes, log2(CPM+1)
mat_cruda = conteos.loc[top10]     # 10 genes x 79 pacientes, conteos crudos

print("Matriz cruda:        ", conteos.shape[0], "genes x", conteos.shape[1], "pacientes")
print("Después del filtro:  ", expr.shape[0], "genes x", expr.shape[1], "pacientes")
print("Matriz reducida:     ", mat_log.shape[0], "genes x", mat_log.shape[1], "pacientes")
print("mat_log.shape =", mat_log.shape)
print("P/N pasa de", round(conteos.shape[0] / conteos.shape[1], 1), "a", round(mat_log.shape[0] / mat_log.shape[1], 2))

nom_top = anot.loc[top10, "gene_name"]
tabla = pd.DataFrame({
    "gen": nom_top,
    "tipo": anot.loc[top10, "gene_type"],
    "varianza": varianza[top10].round(2),
    "media_log2CPM": mat_log.mean(axis=1).round(2),
    "min_log2CPM": mat_log.min(axis=1).round(2),
    "max_log2CPM": mat_log.max(axis=1).round(2),
    "conteo_mediano": mat_cruda.median(axis=1),
})
display(tabla)

# versión con nombres de gen en las filas
vista = mat_log.rename(index=anot["gene_name"].to_dict())
print("\nPrimeros 3 pacientes:")
print(vista.iloc[:, :3].round(2).to_string())

os.makedirs("resultados", exist_ok=True)
vista.to_csv("resultados/matriz_10_genes_log2cpm.csv")
mat_cruda.rename(index=anot["gene_name"].to_dict()).to_csv("resultados/matriz_10_genes_conteos.csv")
tabla.to_csv("resultados/tabla_10_genes.csv")

La reducción va de 60.660 genes a 21.181 después del filtro y a 10 genes al final, manteniendo los 79 pacientes, por lo que la matriz lista para modelar es de 10 x 79 (la relación P/N baja de 767.8 a 0.13). Cabe aclarar que esto resuelve el problema de P >> N, pero no el de tener pocos pacientes: 79 tumores siguen siendo una muestra pequeña para cualquier modelo supervisado.

## 11. Interpretación biológica de los 10 genes

Para interpretar estos genes hay que tener en cuenta cómo funciona la corteza suprarrenal. La corteza produce hormonas esteroideas a partir del colesterol por medio de una cadena de enzimas, y según la zona de la glándula se llega a mineralocorticoides (aldosterona), glucocorticoides (cortisol) o andrógenos. Un ACC funcionante conserva parte de esta maquinaria y secreta hormonas en exceso, mientras que en el estudio de TCGA el puntaje de diferenciación adrenocortical fue más bajo en los tumores menos diferenciados (los dos tumores sarcomatoides tuvieron el más bajo). Entonces los genes de esta vía deberían tener una varianza alta entre pacientes, ya que habría tumores en un extremo y en el otro.

La siguiente celda muestra la distribución de cada gen entre los 79 pacientes (boxplot) y la correlación de Spearman entre los 10 genes. Si la idea anterior es correcta, se esperaría que CYP11B1, HSD3B2 y CYP17A1 estén correlacionados positivamente entre sí, y que FATE1 y GSTA1 se acerquen a ese grupo. Si esto no se observa en el gráfico, la hipótesis del eje de diferenciación esteroidogénica se debilita y la interpretación de abajo debe de revisarse.

In [ ]:
import matplotlib.pyplot as plt

genes = list(vista.index)

fig, ax = plt.subplots(figsize=(9, 4))
ax.boxplot([vista.loc[g].values for g in genes])
ax.set_xticks(range(1, len(genes) + 1))
ax.set_xticklabels(genes, rotation=45, ha="right")
ax.set_ylabel("log2(CPM + 1)")
ax.set_title("Expresión de los 10 genes en los 79 pacientes")
plt.tight_layout()
plt.show()

corr = vista.T.corr(method="spearman")
fig, ax = plt.subplots(figsize=(6.5, 5.5))
im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(genes)))
ax.set_xticklabels(genes, rotation=60, ha="right")
ax.set_yticks(range(len(genes)))
ax.set_yticklabels(genes)
fig.colorbar(im, ax=ax, label="correlación de Spearman")
ax.set_title("Correlación entre los 10 genes (79 pacientes)")
plt.tight_layout()
plt.show()

print(corr.round(2).to_string())

A continuación se describe cada gen en el orden de la varianza.

Con respecto a MEG3 (varianza 15.16, la más alta), este gen no codifica una proteína: es un RNA largo no codificante (lncRNA) ubicado en 14q32.2, dentro del locus impreso DLK1-MEG3, y se expresa solo desde el alelo materno. Se ha descrito que MEG3 activa p53 y que inhibe la proliferación de varias líneas celulares tumorales, que su expresión se pierde en algunos tumores por hipermetilación (por ejemplo, en adenomas hipofisarios no funcionantes) y que se expresa en alto nivel en la glándula suprarrenal normal. Una varianza tan alta indica que hay pacientes que conservan una expresión alta y otros en donde es casi nula, lo cual es compatible con un silenciamiento epigenético en una parte de los tumores, aunque esto es una hipótesis y no se evaluó aquí con los datos de metilación de TCGA-ACC. Además, DLK1, que está en el mismo locus, apareció sexto en la varianza con conteos crudos y décimo en el escenario A, lo que va en la misma dirección.

En cuanto a CYP11B1 (14.82), este gen codifica la 11β-hidroxilasa, una enzima mitocondrial del citocromo P450 que realiza el último paso de la síntesis de cortisol (convierte el 11-desoxicortisol en cortisol) y que se expresa en la zona fascicular de la corteza. En el ACC es relevante porque el exceso de cortisol es la forma más común de hipersecreción (41% de hipercortisolismo aislado en el artículo citado en la sección 1). Su varianza alta sugiere que la cohorte mezcla tumores que mantienen la capacidad de producir cortisol con tumores que no, lo que además se relaciona con el puntaje de diferenciación adrenocortical de TCGA.

El gen GSTA1 (14.70) codifica la glutatión S-transferasa alfa 1, una enzima de detoxificación que conjuga compuestos con glutatión. Se ha relacionado con tejido adrenocortical funcionante: Sarkar et al. (2001) encontraron que GSTA1 estaba sobreexpresado en adenomas benignos de pacientes con síndrome de Cushing y que inhibir la actividad de GST en células H295R (una línea de carcinoma adrenocortical) disminuía su proliferación, por lo que plantean que podría ser una respuesta adaptativa al exceso de producción de cortisol. Es decir, su varianza podría acompañar a la de las enzimas esteroidogénicas, algo que se puede revisar en la matriz de correlación.

Con respecto a HSD3B2 (12.52), este gen codifica la 3β-hidroxiesteroide deshidrogenasa tipo 2, que convierte los esteroides Δ5 (pregnenolona, 17-hidroxipregnenolona y DHEA) en esteroides Δ4 (progesterona, 17-hidroxiprogesterona y androstenediona). Como esta conversión es necesaria para llegar tanto a cortisol y aldosterona como a andrógenos, es uno de los marcadores más directos de que una célula conserva la capacidad de producir esteroides. Una expresión baja en un tumor sugeriría desdiferenciación, y una expresión alta, un tumor que sigue siendo esteroidogénico.

KLK1 (11.50) codifica la calicreína tisular, una serina proteasa que corta el cininógeno para liberar cininas, péptidos que participan en la regulación de la presión arterial y la inflamación. En la literatura que se revisó para este trabajo no se encontró evidencia clara de un papel de KLK1 en el ACC, por lo que su aparición en el top 10 queda como un hallazgo descriptivo: puede reflejar expresión propia del tejido adrenal o del microambiente, pero esto no se puede afirmar con los datos del notebook.

En cuanto a SLPI (11.33), este es el inhibidor de proteasas de leucocitos secretado, que inhibe serina proteasas como la elastasa de neutrófilos y tiene funciones antiinflamatorias y antimicrobianas en mucosas. En algunos tumores sólidos se ha asociado con progresión (favoreciendo la proliferación y la invasión en modelos celulares), pero no se encontró evidencia específica en ACC. Una posible explicación de su variabilidad es el infiltrado inflamatorio de cada tumor, ya que SLPI se produce también en células inmunes y epiteliales, aunque esto es solo una hipótesis.

CYP17A1 (10.75) codifica una sola enzima con dos actividades: la 17α-hidroxilasa (necesaria para llegar al cortisol) y la 17,20-liasa (que produce los precursores de andrógenos como la DHEA). Es el punto en donde la vía se reparte entre glucocorticoides y andrógenos, y por esto es relevante en el ACC: en el artículo de Endocrine el 22% de los pacientes tenía co-secreción de cortisol y andrógenos y el 19% hiperandrogenismo aislado.

Con respecto a FATE1 (10.44), este gen (fetal and adult testis expressed 1) se ha descrito como un antígeno cáncer-testículo, y en el ACC es el que tiene la evidencia más directa de relación con la agresividad. El factor de transcripción SF-1 (el regulador maestro de la esteroidogénesis adrenal) activa su expresión de forma dependiente de la dosis, y la proteína se ubica en la interfaz entre el retículo endoplásmico y la mitocondria, en donde disminuye el contacto entre ambos organelos y la entrada de calcio a la mitocondria, haciendo que la célula resista la apoptosis dependiente de calcio. En una cohorte de 141 ACC, FATE1 se expresó en niveles altos en cerca del 40% de los tumores (los tumores no esteroidogénicos fueron negativos), su disminución aumentó la sensibilidad al mitotano (el fármaco estándar del ACC) en modelos celulares, y una expresión alta se asoció con peor supervivencia en análisis univariado y multivariado (Doghman-Bouguerra et al., EMBO Reports, 2016). Es decir, FATE1 conecta el programa esteroidogénico con la resistencia a la apoptosis y al tratamiento.

CCN3 (10.34), también llamado NOV (nephroblastoma overexpressed), pertenece a la familia CCN de proteínas matricelulares, que se secretan a la matriz extracelular y modulan la adhesión, la proliferación y la diferenciación celular. En la búsqueda bibliográfica de este trabajo no se pudo leer una fuente específica sobre CCN3 en ACC, por lo que no se afirma nada sobre su papel en este tumor; su inclusión se interpreta solo como variabilidad entre pacientes en un gen de matriz extracelular, y el siguiente paso sería buscar estudios en tumores adrenocorticales.

Por último, NPTX1 (10.23) codifica la pentraxina neuronal 1, una proteína sináptica que se expresa sobre todo en el cerebro (corteza cerebral e hipocampo, según el Human Protein Atlas). Su aparición en el top 10 de un tumor adrenal es el hallazgo más difícil de explicar: la misma ficha del Human Protein Atlas incluye a las células de la corteza suprarrenal entre los tipos celulares con expresión enriquecida de este gen, pero también reporta que casi todos los tejidos tumorales son negativos por inmunohistoquímica. Una hipótesis es que su expresión forme parte del programa de diferenciación adrenocortical y varíe con este, y otra es que se deba a unos pocos pacientes con expresión muy alta; la forma de distinguirlas es mirar la distribución por paciente en el boxplot.

En conjunto, tres de los diez genes (CYP11B1, HSD3B2 y CYP17A1) son enzimas directas de la esteroidogénesis, FATE1 es una diana de SF-1 y GSTA1 se ha relacionado con tumores adrenocorticales funcionantes, de modo que una parte importante de la varianza parece estar asociada con el eje de diferenciación esteroidogénica del tumor. MEG3 apunta a otro eje, el epigenético (impronta del locus 14q32). Los otros genes (KLK1, SLPI, CCN3 y NPTX1) no tienen una explicación clara y se dejan como hallazgos descriptivos. La relación de todo esto con la agresividad y la supervivencia viene de la literatura (sobre todo la de FATE1) y no de un análisis hecho con los datos de este notebook.

## 12. Limitaciones y siguientes pasos

Con respecto a la selección, la varianza es un criterio no supervisado: encuentra los genes que más cambian entre pacientes, pero no los que se asocian con la supervivencia, con el estadio o con la agresividad. Además, como la cohorte solo tiene tumores primarios y no tejido normal, los 10 genes no son los que distinguen el tumor del tejido sano, sino los que más varían dentro del tumor. Por esto, que un gen esté en este top 10 no significa que sea un biomarcador: para eso se necesitaría cruzar la matriz con las variables clínicas del GDC (como el estado vital y los días hasta la muerte) y evaluar la asociación con pruebas estadísticas.

Por otra parte, el filtro por varianza usa a todos los pacientes y no usa etiquetas, por lo que el riesgo de leakage es bajo, pero si más adelante se arma un modelo supervisado lo más prudente es repetir la selección dentro de cada fold. También hay que tener en cuenta que los umbrales (10 conteos en el 20% de los pacientes) y la exclusión de los genes del Y son decisiones de preprocesamiento: las secciones 7 y 8 muestran que cambian la cantidad de genes que quedan, y que la lista de 10 depende sobre todo de cómo se normaliza por tamaño de librería. Cabe aclarar que la interpretación biológica de la sección 11 se basa en una revisión bibliográfica limitada y que varias de las fuentes son estudios pequeños o en líneas celulares, por lo que no permiten afirmar causalidad.

## Referencias

- Zheng S, et al. Comprehensive pan-genomic characterization of adrenocortical carcinoma. Cancer Cell, 2016. https://pmc.ncbi.nlm.nih.gov/articles/PMC4864952
- Adrenocortical carcinoma: a population-based study on incidence and survival in the Netherlands since 1993. Eur J Cancer, 2013. https://pubmed.ncbi.nlm.nih.gov/23561851/
- Artículo de Endocrine (Springer) sobre incidencia, secreción hormonal y supervivencia del ACC. DOI 10.1007/s12020-019-01918-9. https://link.springer.com/article/10.1007/s12020-019-01918-9
- Doghman-Bouguerra M, et al. FATE1 antagonizes calcium- and drug-induced apoptosis by uncoupling ER and mitochondria. EMBO Reports, 2016. https://pmc.ncbi.nlm.nih.gov/articles/PMC5007562
- Sarkar D, et al. Overexpression of glutathione-S-transferase A1 in benign adrenocortical adenomas from patients with Cushing's syndrome. J Clin Endocrinol Metab, 2001; 86(4): 1653-1659.
- Atlas of Genetics and Cytogenetics in Oncology and Haematology, entrada MEG3. https://atlasgeneticsoncology.org/gene/44393/meg3-(maternally-expressed-3)
- Human Protein Atlas, NPTX1. https://www.proteinatlas.org/Q15818
- Genomic Data Commons. https://portal.gdc.cancer.gov y https://api.gdc.cancer.gov